# Cross-Run Integration Master Notebook

This notebook is the final master integration step for Axis 2 + cross-axis bridge.

It does three things:
1. Builds a cross-run comparison table across all configured run tags.
2. Aggregates per-bit AUROC outputs across runs.
3. Joins run-wise per-bit AUROC with cross-axis correlation prep outputs (descriptor matches).

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

In [ ]:
# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
ROOT = Path('..')
MODEL_RUNS_DIR = ROOT / 'results' / 'model_runs'
CROSS_AXIS_DIR = ROOT / 'results' / 'cross_axis_correlation_prep'
OUT_DIR = ROOT / 'results' / 'cross_run_integration'
FIGURES_DIR = ROOT / 'results' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Configure the runs you want to include in the master comparison.
RUN_TAGS = [
    'morgan_2048_cos',
    'morgan_2048_bce',
    'maccs_166_cos',
    'maccs_166_bce',
    'map4_2048_cos',
    'map4_2048_bce',
    'morgan_2048_bce_frozen',
    'maccs_166_bce_frozen',
    'map4_2048_bce_frozen',
    # Optional:
    'morgan_2048_cos_frozen',
    'maccs_166_cos_frozen',
    'map4_2048_cos_frozen',
]

print('MODEL_RUNS_DIR:', MODEL_RUNS_DIR)
print('CROSS_AXIS_DIR:', CROSS_AXIS_DIR)
print('OUT_DIR:', OUT_DIR)
print('FIGURES_DIR:', FIGURES_DIR)
print('Configured run tags:', len(RUN_TAGS))


In [ ]:
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def parse_run_tag(run_tag):
    frozen = run_tag.endswith('_frozen')
    base = run_tag[:-7] if frozen else run_tag

    if base.endswith('_bce'):
        loss = 'bce'
        fp_token = base[:-4]
    elif base.endswith('_cos'):
        loss = 'cos'
        fp_token = base[:-4]
    else:
        loss = 'unknown'
        fp_token = base

    fp_map = {
        'morgan_2048': 'ecfp4',
        'maccs_166': 'maccs',
        'map4_1024': 'map4',
        'map4_2048': 'map4',
    }
    fp_family = fp_map.get(fp_token, fp_token)

    return {
        'run_tag': run_tag,
        'fp_token': fp_token,
        'fp_family': fp_family,
        'loss_kind': loss,
        'is_frozen': frozen,
    }


def load_metric_table(path):
    if not path.exists():
        return None
    return pd.read_csv(path)


def metric_lookup(metric_map, *keys):
    for key in keys:
        val = metric_map.get(key, np.nan)
        if pd.notna(val):
            return val
    return np.nan


def compute_rowwise_cosine_stats(y_pred, y_true):
    numer = np.sum(y_pred * y_true, axis=1)
    denom = np.linalg.norm(y_pred, axis=1) * np.linalg.norm(y_true, axis=1)
    sims = numer / np.maximum(denom, 1e-8)
    return {
        'cosine_sim_mean': float(np.mean(sims)),
        'cosine_sim_median': float(np.median(sims)),
    }


def load_cosine_metrics_from_artifacts(artifacts_dir):
    paths = {
        'y_pred_ood': artifacts_dir / 'y_pred.npy',
        'y_true_ood': artifacts_dir / 'y_true.npy',
        'y_pred_val': artifacts_dir / 'y_pred_val.npy',
        'y_true_val': artifacts_dir / 'y_true_val.npy',
    }
    if any(not p.exists() for p in paths.values()):
        return {}

    y_pred_val = np.load(paths['y_pred_val'])
    y_true_val = np.load(paths['y_true_val'])
    y_pred_ood = np.load(paths['y_pred_ood'])
    y_true_ood = np.load(paths['y_true_ood'])

    val_metrics = compute_rowwise_cosine_stats(y_pred_val, y_true_val)
    ood_metrics = compute_rowwise_cosine_stats(y_pred_ood, y_true_ood)
    mean_drop = val_metrics['cosine_sim_mean'] - ood_metrics['cosine_sim_mean']
    median_drop = val_metrics['cosine_sim_median'] - ood_metrics['cosine_sim_median']
    return {
        'cosine_sim_mean_val': val_metrics['cosine_sim_mean'],
        'cosine_sim_mean_ood': ood_metrics['cosine_sim_mean'],
        'cosine_sim_mean_drop': float(mean_drop),
        'cosine_gap': float(mean_drop),
        'cosine_sim_median_val': val_metrics['cosine_sim_median'],
        'cosine_sim_median_ood': ood_metrics['cosine_sim_median'],
        'cosine_sim_median_drop': float(median_drop),
    }


def retrieval_metrics_to_dict(df):
    if df is None or len(df) == 0:
        return {}
    if not {'Metric', 'Validation', 'OOD'}.issubset(df.columns):
        return {}

    out = {}
    for _, r in df.iterrows():
        key = str(r['Metric']).strip().lower().replace(' ', '_')
        key = key.replace('@', 'at').replace('-', '_')
        out[f'{key}_val'] = r['Validation']
        out[f'{key}_ood'] = r['OOD']
    return out


In [ ]:
# ------------------------------------------------------------------
# Build run-level comparison table
# ------------------------------------------------------------------
run_rows = []
missing_runs = []

for run_tag in RUN_TAGS:
    info = parse_run_tag(run_tag)
    artifacts_dir = MODEL_RUNS_DIR / run_tag / 'axis2_artifacts'

    if not artifacts_dir.exists():
        missing_runs.append(run_tag)
        continue

    auroc_summary = load_metric_table(artifacts_dir / 'per_bit_auroc' / 'auroc_summary_table.csv')
    threshold_cmp = load_metric_table(artifacts_dir / 'threshold_sweep' / 'threshold_comparison_table.csv')
    retrieval_metrics = load_metric_table(artifacts_dir / 'retrieval' / 'retrieval_metrics.csv')
    run_cfg = artifacts_dir / 'run_config.json'

    row = {
        **info,
        'artifacts_dir': str(artifacts_dir),
        'has_run_config': run_cfg.exists(),
        'has_auroc_summary': auroc_summary is not None,
        'has_threshold_summary': threshold_cmp is not None,
        'has_retrieval_summary': retrieval_metrics is not None,
    }

    if auroc_summary is not None and {'metric', 'value'}.issubset(auroc_summary.columns):
        metric_map = dict(zip(auroc_summary['metric'], auroc_summary['value']))
        row['n_bits'] = metric_lookup(metric_map, 'n_bits')
        row['global_auroc_val'] = metric_lookup(metric_map, 'global_auroc_val')
        row['global_auroc_ood'] = metric_lookup(metric_map, 'global_auroc_ood')
        row['mean_per_bit_auroc_val'] = metric_lookup(metric_map, 'mean_per_bit_auroc_val', 'mean_auroc_val')
        row['mean_per_bit_auroc_ood'] = metric_lookup(metric_map, 'mean_per_bit_auroc_ood', 'mean_auroc_ood')
        row['mean_per_bit_auroc_drop'] = metric_lookup(metric_map, 'mean_per_bit_auroc_drop', 'mean_auroc_drop')

        # Backward-compatible aliases.
        row['mean_auroc_val'] = row['mean_per_bit_auroc_val']
        row['mean_auroc_ood'] = row['mean_per_bit_auroc_ood']
        row['mean_auroc_drop'] = row['mean_per_bit_auroc_drop']

    if threshold_cmp is not None and {'split', 'best_tau_tanimoto', 'best_tanimoto'}.issubset(threshold_cmp.columns):
        val_row = threshold_cmp[threshold_cmp['split'] == 'val']
        ood_row = threshold_cmp[threshold_cmp['split'] == 'ood']
        if len(val_row):
            row['best_tau_val'] = float(val_row['best_tau_tanimoto'].iloc[0])
            row['best_tanimoto_val'] = float(val_row['best_tanimoto'].iloc[0])
        if len(ood_row):
            row['best_tau_ood'] = float(ood_row['best_tau_tanimoto'].iloc[0])
            row['best_tanimoto_ood'] = float(ood_row['best_tanimoto'].iloc[0])

    row.update(retrieval_metrics_to_dict(retrieval_metrics))

    cosine_missing = (
        pd.isna(row.get('cosine_sim_mean_val', np.nan))
        or pd.isna(row.get('cosine_sim_mean_ood', np.nan))
    )
    if cosine_missing:
        row.update(load_cosine_metrics_from_artifacts(artifacts_dir))

    if pd.notna(row.get('cosine_sim_mean_val', np.nan)) and pd.notna(row.get('cosine_sim_mean_ood', np.nan)):
        row['cosine_sim_mean_drop'] = float(row['cosine_sim_mean_val']) - float(row['cosine_sim_mean_ood'])
        row['cosine_gap'] = row['cosine_sim_mean_drop']
    if pd.notna(row.get('cosine_sim_median_val', np.nan)) and pd.notna(row.get('cosine_sim_median_ood', np.nan)):
        row['cosine_sim_median_drop'] = float(row['cosine_sim_median_val']) - float(row['cosine_sim_median_ood'])

    run_rows.append(row)

run_summary_df = pd.DataFrame(run_rows)
if len(run_summary_df):
    run_summary_df = run_summary_df.sort_values(['fp_family', 'loss_kind', 'is_frozen', 'run_tag']).reset_index(drop=True)
    run_summary_df['model_family'] = np.where(run_summary_df['is_frozen'], 'Frozen', 'Fine-tuned')
    run_summary_df['fp_display'] = run_summary_df['fp_family'].map({
        'maccs': 'MACCS',
        'ecfp4': 'Morgan / ECFP4',
        'map4': 'MAP4',
    }).fillna(run_summary_df['fp_family'])
    run_summary_df['loss_display'] = run_summary_df['loss_kind'].str.upper()

print('Available runs loaded:', len(run_summary_df))
if missing_runs:
    print('Missing run folders:')
    for x in missing_runs:
        print('  -', x)

display(run_summary_df)
run_summary_df.to_csv(OUT_DIR / 'cross_run_summary_table.csv', index=False)
print('Saved:', OUT_DIR / 'cross_run_summary_table.csv')


In [13]:
# ------------------------------------------------------------------
# Collect per-bit AUROC across all runs
# ------------------------------------------------------------------
bit_rows = []
for _, rr in run_summary_df.iterrows():
    run_tag = rr['run_tag']
    fp_family = rr['fp_family']
    loss_kind = rr['loss_kind']
    is_frozen = bool(rr['is_frozen'])
    auroc_path = MODEL_RUNS_DIR / run_tag / 'axis2_artifacts' / 'per_bit_auroc' / 'auroc_comparison.csv'

    if not auroc_path.exists():
        continue

    dfb = pd.read_csv(auroc_path)
    required = {'bit_index', 'auroc_val', 'auroc_ood', 'auroc_drop'}
    if not required.issubset(dfb.columns):
        continue

    dfb = dfb[['bit_index', 'auroc_val', 'auroc_ood', 'auroc_drop']].copy()
    dfb['run_tag'] = run_tag
    dfb['fp_family'] = fp_family
    dfb['loss_kind'] = loss_kind
    dfb['is_frozen'] = is_frozen
    bit_rows.append(dfb)

if bit_rows:
    per_bit_all = pd.concat(bit_rows, ignore_index=True)
else:
    per_bit_all = pd.DataFrame(columns=['bit_index', 'auroc_val', 'auroc_ood', 'auroc_drop', 'run_tag', 'fp_family', 'loss_kind', 'is_frozen'])

display(per_bit_all.head())
per_bit_all.to_csv(OUT_DIR / 'per_bit_auroc_all_runs_long.csv', index=False)
print('Saved:', OUT_DIR / 'per_bit_auroc_all_runs_long.csv')

,bit_index,auroc_val,auroc_ood,auroc_drop,run_tag,fp_family,loss_kind,is_frozen
0,0,0.851001,0.589763,0.261237,morgan_2048_bce,ecfp4,bce,False
1,1,0.856268,0.652011,0.204257,morgan_2048_bce,ecfp4,bce,False
2,2,0.766053,0.685786,0.080267,morgan_2048_bce,ecfp4,bce,False
3,3,0.817669,0.693223,0.124446,morgan_2048_bce,ecfp4,bce,False
4,4,0.801978,0.551487,0.250491,morgan_2048_bce,ecfp4,bce,False


Saved: ../results/cross_run_integration/per_bit_auroc_all_runs_long.csv


In [14]:
# ------------------------------------------------------------------
# Join with cross-axis correlation prep outputs
# ------------------------------------------------------------------
match_files = {
    'ecfp4': CROSS_AXIS_DIR / 'best_matches_ecfp4.csv',
    'maccs': CROSS_AXIS_DIR / 'best_matches_maccs.csv',
    'map4': CROSS_AXIS_DIR / 'best_matches_map4.csv',
}

joined_rows = []
for fp_family, mpath in match_files.items():
    if not mpath.exists():
        print(f'Skip missing cross-axis file for {fp_family}: {mpath}')
        continue

    mdf = pd.read_csv(mpath)
    if 'bit_index' not in mdf.columns:
        continue

    pb = per_bit_all[per_bit_all['fp_family'] == fp_family].copy()
    if len(pb) == 0:
        continue

    merged = pb.merge(mdf, on='bit_index', how='left', suffixes=('', '_corr'))
    joined_rows.append(merged)

if joined_rows:
    cross_axis_join = pd.concat(joined_rows, ignore_index=True)
else:
    cross_axis_join = pd.DataFrame()

display(cross_axis_join.head())
cross_axis_join.to_csv(OUT_DIR / 'cross_axis_join_all_runs.csv', index=False)
print('Saved:', OUT_DIR / 'cross_axis_join_all_runs.csv')

if len(cross_axis_join):
    top_desc = (
        cross_axis_join
        .dropna(subset=['best_descriptor'])
        .groupby(['run_tag', 'fp_family', 'best_descriptor'], as_index=False)
        .agg(
            mean_auroc_val=('auroc_val', 'mean'),
            mean_auroc_ood=('auroc_ood', 'mean'),
            mean_auroc_drop=('auroc_drop', 'mean'),
            n_bits=('bit_index', 'count')
        )
        .sort_values(['run_tag', 'n_bits', 'mean_auroc_ood'], ascending=[True, False, False])
    )
    top_desc.to_csv(OUT_DIR / 'descriptor_level_summary_by_run.csv', index=False)
    print('Saved:', OUT_DIR / 'descriptor_level_summary_by_run.csv')

,bit_index,auroc_val,auroc_ood,auroc_drop,run_tag,fp_family,loss_kind,is_frozen,best_descriptor_idx,best_descriptor,best_corr,best_abs_corr
0,0,0.851001,0.589763,0.261237,morgan_2048_bce,ecfp4,bce,False,170.0,fr_lactone,0.109718,0.109718
1,1,0.856268,0.652011,0.204257,morgan_2048_bce,ecfp4,bce,False,66.0,NumRotatableBonds,0.418655,0.418655
2,2,0.766053,0.685786,0.080267,morgan_2048_bce,ecfp4,bce,False,185.0,fr_piperdine,0.160847,0.160847
3,3,0.817669,0.693223,0.124446,morgan_2048_bce,ecfp4,bce,False,188.0,fr_pyridine,0.114462,0.114462
4,4,0.801978,0.551487,0.250491,morgan_2048_bce,ecfp4,bce,False,191.0,fr_sulfonamd,0.153722,0.153722


Saved: ../results/cross_run_integration/cross_axis_join_all_runs.csv
Saved: ../results/cross_run_integration/descriptor_level_summary_by_run.csv


In [ ]:
# ------------------------------------------------------------------
# Visualizations: plotting prep + diagnostic by-run view
# ------------------------------------------------------------------
metric_col_val = 'mean_per_bit_auroc_val' if 'mean_per_bit_auroc_val' in run_summary_df.columns else 'mean_auroc_val'
metric_col_ood = 'mean_per_bit_auroc_ood' if 'mean_per_bit_auroc_ood' in run_summary_df.columns else 'mean_auroc_ood'
plot_df = pd.DataFrame()
plot_long = pd.DataFrame()

if len(run_summary_df) > 0 and {metric_col_val, metric_col_ood}.issubset(run_summary_df.columns):
    plot_df = run_summary_df.dropna(subset=[metric_col_val, metric_col_ood]).copy()

if len(plot_df):
    order = plot_df.sort_values(metric_col_ood, ascending=False)['run_tag']
    plot_long = plot_df.melt(
        id_vars=['run_tag', 'is_frozen', 'fp_family', 'loss_kind', 'model_family', 'fp_display', 'loss_display'],
        value_vars=[metric_col_val, metric_col_ood],
        var_name='split',
        value_name='mean_auroc'
    )
    split_label_map = {metric_col_val: 'Val', metric_col_ood: 'OOD'}
    plot_long['split'] = plot_long['split'].map(split_label_map)

    plt.figure(figsize=(14, 6))
    sns.barplot(
        data=plot_long,
        x='run_tag',
        y='mean_auroc',
        hue='split',
        order=order,
        hue_order=['Val', 'OOD']
    )
    plt.xticks(rotation=45, ha='right')
    plt.title('Mean Per-Bit AUROC by Run (Val vs OOD)')
    plt.ylabel('Mean AUROC')
    plt.xlabel('Run Tag')
    plt.tight_layout()
    legacy_png = OUT_DIR / 'mean_auroc_val_vs_ood_by_run.png'
    plt.savefig(legacy_png, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', legacy_png)
else:
    print('No AUROC summary data available for plotting.')


In [ ]:
# ------------------------------------------------------------------
# Figure: Fine-tuned vs frozen by fingerprint family and loss
# ------------------------------------------------------------------
if len(plot_df):
    fp_order = [x for x in ['MACCS', 'Morgan / ECFP4', 'MAP4'] if x in set(plot_df['fp_display'])]
    facet_df = plot_long.copy()
    facet_df['split'] = pd.Categorical(facet_df['split'], categories=['Val', 'OOD'], ordered=True)
    facet_df['model_family'] = pd.Categorical(facet_df['model_family'], categories=['Fine-tuned', 'Frozen'], ordered=True)
    facet_df['loss_display'] = pd.Categorical(facet_df['loss_display'], categories=['BCE', 'COS'], ordered=True)

    g = sns.catplot(
        data=facet_df,
        kind='bar',
        x='loss_display',
        y='mean_auroc',
        hue='model_family',
        row='split',
        col='fp_display',
        row_order=['Val', 'OOD'],
        col_order=fp_order,
        order=['BCE', 'COS'],
        hue_order=['Fine-tuned', 'Frozen'],
        palette={'Fine-tuned': '#4C72B0', 'Frozen': '#DD8452'},
        height=3.7,
        aspect=0.95,
        sharey=True,
        legend=True,
    )
    g.set_axis_labels('Loss', 'Mean Per-Bit AUROC')
    g.set_titles('{row_name} | {col_name}')
    g.figure.subplots_adjust(top=0.88)
    g.figure.suptitle('Fine-Tuned vs Frozen by Fingerprint Family and Loss', fontsize=15)
    for ax in g.axes.flat:
        for container in ax.containers:
            try:
                ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
            except Exception:
                pass

    analysis_png = OUT_DIR / 'fine_tuned_vs_frozen_by_fp_and_loss.png'
    figure_png = FIGURES_DIR / 'fine_tuned_vs_frozen_by_fp_and_loss.png'
    figure_pdf = FIGURES_DIR / 'fine_tuned_vs_frozen_by_fp_and_loss.pdf'
    g.savefig(analysis_png, dpi=200, bbox_inches='tight')
    g.savefig(figure_png, dpi=300, bbox_inches='tight')
    g.savefig(figure_pdf, bbox_inches='tight')
    plt.show()
    print('Saved:', analysis_png)
    print('Saved:', figure_png)
    print('Saved:', figure_pdf)
else:
    print('Skipping fine-tuned vs frozen figure: no plot data.')


In [ ]:
# ------------------------------------------------------------------
# Figure: Fine-tuning gain heatmap + diagnostic table
# ------------------------------------------------------------------
if len(plot_df):
    pair_rows = []
    for (fp_display, loss_display), grp in plot_df.groupby(['fp_display', 'loss_display'], sort=False):
        fine = grp[grp['model_family'] == 'Fine-tuned']
        frozen = grp[grp['model_family'] == 'Frozen']
        if len(fine) and len(frozen):
            pair_rows.append({
                'fp_display': fp_display,
                'loss_display': loss_display,
                'fine_minus_frozen_val': float(fine[metric_col_val].iloc[0] - frozen[metric_col_val].iloc[0]),
                'fine_minus_frozen_ood': float(fine[metric_col_ood].iloc[0] - frozen[metric_col_ood].iloc[0]),
            })

    if pair_rows:
        pair_df = pd.DataFrame(pair_rows)
        pair_df = pair_df.sort_values(['fp_display', 'loss_display']).reset_index(drop=True)
        pair_csv = OUT_DIR / 'fine_tuned_vs_frozen_delta_by_fp_and_loss.csv'
        pair_df.to_csv(pair_csv, index=False)
        display(pair_df)

        heat_df = pair_df.set_index(['fp_display', 'loss_display'])[
            ['fine_minus_frozen_val', 'fine_minus_frozen_ood']
        ]
        heat_df.columns = ['Val', 'OOD']

        plt.figure(figsize=(6.5, 4.5))
        sns.heatmap(
            heat_df,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            cbar_kws={'label': 'Fine-tuned minus Frozen AUROC'}
        )
        plt.title('Fine-Tuned Minus Frozen AUROC by Fingerprint and Loss')
        plt.xlabel('Split')
        plt.ylabel('Fingerprint | Loss')
        plt.tight_layout()

        analysis_png = OUT_DIR / 'fine_tuned_minus_frozen_heatmap.png'
        figure_png = FIGURES_DIR / 'fine_tuned_minus_frozen_heatmap.png'
        figure_pdf = FIGURES_DIR / 'fine_tuned_minus_frozen_heatmap.pdf'
        plt.savefig(analysis_png, dpi=200, bbox_inches='tight')
        plt.savefig(figure_png, dpi=300, bbox_inches='tight')
        plt.savefig(figure_pdf, bbox_inches='tight')
        plt.show()
        print('Saved:', analysis_png)
        print('Saved:', figure_png)
        print('Saved:', figure_pdf)
        print('Saved:', pair_csv)

    diag = plot_df[['run_tag', 'fp_display', 'loss_display', 'model_family', metric_col_val, metric_col_ood]].copy()
    diag = diag.rename(columns={
        metric_col_val: 'mean_auroc_val',
        metric_col_ood: 'mean_auroc_ood',
    })
    diag['ood_minus_val'] = diag['mean_auroc_ood'] - diag['mean_auroc_val']
    diag = diag.sort_values('ood_minus_val', ascending=False).reset_index(drop=True)
    display(diag)
    diag_csv = OUT_DIR / 'val_vs_ood_gap_by_run.csv'
    diag.to_csv(diag_csv, index=False)
    print('Saved:', diag_csv)
else:
    print('Skipping delta heatmap and diagnostic table: no plot data.')

# ------------------------------------------------------------------
# Comparative tables for narrative questions
# ------------------------------------------------------------------
def safe_delta(a, b):
    if pd.notna(a) and pd.notna(b):
        return float(a - b)
    return np.nan


def classify_delta_magnitude(delta_cos, delta_auc, delta_acc1):
    abs_cos = abs(delta_cos)
    abs_auc = abs(delta_auc)
    abs_acc1 = abs(delta_acc1)
    if abs_cos <= 0.01 and abs_auc <= 0.02 and abs_acc1 <= 0.005:
        return 'Small'
    if abs_cos >= 0.02 or abs_auc >= 0.05 or abs_acc1 >= 0.01:
        return 'Large'
    return 'Moderate'


def classify_delta_direction(*vals):
    pos = sum(v > 0 for v in vals if pd.notna(v))
    neg = sum(v < 0 for v in vals if pd.notna(v))
    n = sum(pd.notna(v) for v in vals)
    if n == 0:
        return 'Unknown'
    if pos == n:
        return 'Positive'
    if neg == n:
        return 'Negative'
    return 'Mixed'


if len(run_summary_df):
    metric_pairs = {
        'cosine_sim_mean': ('cosine_sim_mean_val', 'cosine_sim_mean_ood'),
        'mean_per_bit_auroc': (metric_col_val, metric_col_ood),
        'accat1': ('accat1_val', 'accat1_ood'),
        'best_tanimoto': ('best_tanimoto_val', 'best_tanimoto_ood'),
    }

    delta_rows = []
    for (fp_display, loss_display), grp in run_summary_df.groupby(['fp_display', 'loss_display'], sort=True):
        fine = grp[grp['model_family'] == 'Fine-tuned']
        frozen = grp[grp['model_family'] == 'Frozen']
        if not len(fine) or not len(frozen):
            continue

        fine = fine.iloc[0]
        frozen = frozen.iloc[0]
        row = {
            'fp_display': fp_display,
            'loss_display': loss_display,
            'fine_tuned_run_tag': fine['run_tag'],
            'frozen_run_tag': frozen['run_tag'],
        }
        for metric_name, (val_col, ood_col) in metric_pairs.items():
            row[f'delta_{metric_name}_val'] = safe_delta(fine.get(val_col), frozen.get(val_col))
            row[f'delta_{metric_name}_ood'] = safe_delta(fine.get(ood_col), frozen.get(ood_col))

        row['delta_magnitude_ood'] = classify_delta_magnitude(
            row['delta_cosine_sim_mean_ood'],
            row['delta_mean_per_bit_auroc_ood'],
            row['delta_accat1_ood'],
        )
        row['delta_direction_ood'] = classify_delta_direction(
            row['delta_cosine_sim_mean_ood'],
            row['delta_mean_per_bit_auroc_ood'],
            row['delta_accat1_ood'],
        )
        if row['delta_magnitude_ood'] == 'Small':
            row['interpretation_ood'] = 'SSL mostly sufficient'
        elif row['delta_direction_ood'] == 'Positive':
            row['interpretation_ood'] = 'Fine-tuning adds value'
        else:
            row['interpretation_ood'] = 'Task-dependent reorganisation'
        delta_rows.append(row)

    fine_tuning_delta_df = pd.DataFrame(delta_rows).sort_values(['fp_display', 'loss_display']).reset_index(drop=True)
    fine_tuning_delta_csv = OUT_DIR / 'fine_tuned_vs_frozen_delta_table.csv'
    fine_tuning_delta_df.to_csv(fine_tuning_delta_csv, index=False)
    display(fine_tuning_delta_df)
    print('Saved:', fine_tuning_delta_csv)

    delta_metric_specs = [
        ('delta_cosine_sim_mean', 'Cosine Similarity'),
        ('delta_mean_per_bit_auroc', 'Mean Per-Bit AUROC'),
        ('delta_accat1', 'Retrieval Acc@1'),
    ]
    fp_axis_map = {
        'MACCS': 'MACCS',
        'Morgan / ECFP4': 'ECFP4',
        'MAP4': 'MAP4',
    }
    loss_label_map = {
        'BCE': 'BCE',
        'COS': 'Cosine',
    }
    delta_plot_rows = []
    for _, r in fine_tuning_delta_df.iterrows():
        for split_suffix, split_display in [('val', 'Val'), ('ood', 'OOD')]:
            for metric_prefix, metric_display in delta_metric_specs:
                delta_plot_rows.append({
                    'fp_display': r['fp_display'],
                    'fp_axis_label': fp_axis_map.get(r['fp_display'], r['fp_display']),
                    'loss_display': r['loss_display'],
                    'loss_label': loss_label_map.get(r['loss_display'], r['loss_display']),
                    'condition_label': f"{fp_axis_map.get(r['fp_display'], r['fp_display'])} | {loss_label_map.get(r['loss_display'], r['loss_display'])}",
                    'split': split_display,
                    'metric_display': metric_display,
                    'delta_value': r[f'{metric_prefix}_{split_suffix}'],
                    'fine_tuned_run_tag': r['fine_tuned_run_tag'],
                    'frozen_run_tag': r['frozen_run_tag'],
                })

    fine_tuning_delta_plot_df = pd.DataFrame(delta_plot_rows)
    delta_plot_csv = OUT_DIR / 'fine_tuned_vs_frozen_delta_figure_data.csv'
    fine_tuning_delta_plot_df.to_csv(delta_plot_csv, index=False)
    print('Saved:', delta_plot_csv)

    if len(fine_tuning_delta_plot_df):
        fp_order = [x for x in ['MACCS', 'ECFP4', 'MAP4'] if x in set(fine_tuning_delta_plot_df['fp_axis_label'])]
        fine_tuning_delta_plot_df['split'] = pd.Categorical(fine_tuning_delta_plot_df['split'], categories=['Val', 'OOD'], ordered=True)
        fine_tuning_delta_plot_df['metric_display'] = pd.Categorical(
            fine_tuning_delta_plot_df['metric_display'],
            categories=['Cosine Similarity', 'Mean Per-Bit AUROC', 'Retrieval Acc@1'],
            ordered=True,
        )
        fine_tuning_delta_plot_df['loss_label'] = pd.Categorical(
            fine_tuning_delta_plot_df['loss_label'],
            categories=['BCE', 'Cosine'],
            ordered=True,
        )

        metric_limits = {}
        for metric_name, metric_df in fine_tuning_delta_plot_df.groupby('metric_display', observed=False):
            vals = metric_df['delta_value'].astype(float)
            lim = max(0.005, float(np.nanmax(np.abs(vals)))) * 1.18
            metric_limits[str(metric_name)] = lim

        caption_text = (
            'Delta = fine-tuned - frozen; positive values indicate that fine-tuning improves over frozen SSL embeddings. '
            'Y-axis limits are shared within each metric column to compare Val vs OOD gains directly.'
        )
        caption_path = OUT_DIR / 'fine_tuned_vs_frozen_delta_key_metrics_caption.txt'
        caption_path.write_text(caption_text + '\n')
        print('Saved:', caption_path)

        g = sns.catplot(
            data=fine_tuning_delta_plot_df,
            kind='bar',
            x='fp_axis_label',
            y='delta_value',
            hue='loss_label',
            row='split',
            col='metric_display',
            row_order=['Val', 'OOD'],
            col_order=['Cosine Similarity', 'Mean Per-Bit AUROC', 'Retrieval Acc@1'],
            order=fp_order,
            hue_order=['BCE', 'Cosine'],
            palette={'BCE': '#163A70', 'Cosine': '#E5895E'},
            height=3.8,
            aspect=1.05,
            sharey=False,
            legend=True,
        )
        g.set_axis_labels('Fingerprint Family', 'Fine-tuned minus Frozen')
        g.set_titles('{row_name} | {col_name}')
        g.figure.subplots_adjust(top=0.86, bottom=0.14)
        g.figure.suptitle('Frozen vs Fine-Tuned Delta on Key Metrics', fontsize=15)
        if g._legend is not None:
            g._legend.set_title('Loss')
        for col_idx, metric_name in enumerate(['Cosine Similarity', 'Mean Per-Bit AUROC', 'Retrieval Acc@1']):
            ylim = metric_limits[metric_name]
            for row_idx in range(len(g.axes)):
                g.axes[row_idx, col_idx].set_ylim(-ylim, ylim)
        for ax in g.axes.flat:
            ax.axhline(0.0, color='black', linewidth=1.2, alpha=0.8)
            for container in ax.containers:
                try:
                    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
                except Exception:
                    pass
        g.figure.text(0.5, 0.04, caption_text, ha='center', va='center', fontsize=10)

        analysis_png = OUT_DIR / 'fine_tuned_vs_frozen_delta_key_metrics.png'
        figure_png = FIGURES_DIR / 'fine_tuned_vs_frozen_delta_key_metrics.png'
        figure_pdf = FIGURES_DIR / 'fine_tuned_vs_frozen_delta_key_metrics.pdf'
        g.savefig(analysis_png, dpi=200, bbox_inches='tight')
        g.savefig(figure_png, dpi=300, bbox_inches='tight')
        g.savefig(figure_pdf, bbox_inches='tight')
        plt.show()
        print('Saved:', analysis_png)
        print('Saved:', figure_png)
        print('Saved:', figure_pdf)
    else:
        print('Skipping frozen-vs-fine-tuned delta figure: no delta rows.')

    loss_rows = []
    for (fp_display, model_family), grp in run_summary_df.groupby(['fp_display', 'model_family'], sort=True):
        bce = grp[grp['loss_display'] == 'BCE']
        cos = grp[grp['loss_display'] == 'COS']
        if not len(bce) or not len(cos):
            continue

        bce = bce.iloc[0]
        cos = cos.iloc[0]
        row = {
            'fp_display': fp_display,
            'model_family': model_family,
            'bce_run_tag': bce['run_tag'],
            'cos_run_tag': cos['run_tag'],
        }
        for metric_name, (val_col, ood_col) in metric_pairs.items():
            row[f'bce_minus_cos_{metric_name}_val'] = safe_delta(bce.get(val_col), cos.get(val_col))
            row[f'bce_minus_cos_{metric_name}_ood'] = safe_delta(bce.get(ood_col), cos.get(ood_col))
        row['bce_improves_tanimoto_ood'] = bool(row['bce_minus_cos_best_tanimoto_ood'] > 0)
        loss_rows.append(row)

    loss_comparison_df = pd.DataFrame(loss_rows).sort_values(['fp_display', 'model_family']).reset_index(drop=True)
    loss_comparison_csv = OUT_DIR / 'bce_vs_cos_comparison_table.csv'
    loss_comparison_df.to_csv(loss_comparison_csv, index=False)
    display(loss_comparison_df)
    print('Saved:', loss_comparison_csv)

    fingerprint_rows = []
    ref_fp = 'Morgan / ECFP4'
    for (loss_display, model_family), grp in run_summary_df.groupby(['loss_display', 'model_family'], sort=True):
        ref = grp[grp['fp_display'] == ref_fp]
        if not len(ref):
            continue
        ref = ref.iloc[0]

        for comp_fp in ['MACCS', 'MAP4']:
            comp = grp[grp['fp_display'] == comp_fp]
            if not len(comp):
                continue
            comp = comp.iloc[0]
            row = {
                'comparison_fp_display': comp_fp,
                'reference_fp_display': ref_fp,
                'loss_display': loss_display,
                'model_family': model_family,
                'comparison_run_tag': comp['run_tag'],
                'reference_run_tag': ref['run_tag'],
            }
            for metric_name, (val_col, ood_col) in metric_pairs.items():
                row[f'delta_{metric_name}_val'] = safe_delta(comp.get(val_col), ref.get(val_col))
                row[f'delta_{metric_name}_ood'] = safe_delta(comp.get(ood_col), ref.get(ood_col))
            fingerprint_rows.append(row)

    fingerprint_comparison_df = pd.DataFrame(fingerprint_rows).sort_values(['comparison_fp_display', 'loss_display', 'model_family']).reset_index(drop=True)
    fingerprint_comparison_csv = OUT_DIR / 'fingerprint_family_comparison_table.csv'
    fingerprint_comparison_df.to_csv(fingerprint_comparison_csv, index=False)
    display(fingerprint_comparison_df)
    print('Saved:', fingerprint_comparison_csv)

    objective_specs = [
        ('Best Retrieval (OOD Acc@1)', 'accat1_ood'),
        ('Best Binarized Fingerprint (OOD Tanimoto)', 'best_tanimoto_ood'),
        ('Best Mean Per-Bit AUROC (OOD)', metric_col_ood),
        ('Best Cosine Similarity (OOD)', 'cosine_sim_mean_ood'),
    ]
    best_rows = []
    for objective_label, metric_col in objective_specs:
        subset = run_summary_df.dropna(subset=[metric_col]).sort_values(metric_col, ascending=False)
        if not len(subset):
            continue
        best = subset.iloc[0]
        best_rows.append({
            'objective': objective_label,
            'metric_col': metric_col,
            'best_run_tag': best['run_tag'],
            'fp_display': best['fp_display'],
            'loss_display': best['loss_display'],
            'model_family': best['model_family'],
            'metric_value': float(best[metric_col]),
        })

    best_objective_df = pd.DataFrame(best_rows)
    best_objective_csv = OUT_DIR / 'best_run_by_objective_table.csv'
    best_objective_df.to_csv(best_objective_csv, index=False)
    display(best_objective_df)
    print('Saved:', best_objective_csv)
else:
    print('Skipping comparative tables: no run summary data.')

print('Master integration complete.')
